In [ ]:
from pathlib import Path
import hashlib
import warnings

import streamlit as st
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import StreamlitChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoTokenizer, pipeline
import os


warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
os.environ["HF_TOKEN"] = ""

BASE_DIR = Path(__file__).resolve().parent
PDF_DIR = BASE_DIR / "RAG FILES"
CHROMA_DIR = BASE_DIR / "chroma_db"
COLLECTION_NAME = "rag_pdf_collection"
LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"


st.set_page_config(page_title="RAG com PDFs", page_icon="📚", layout="centered")
st.title("Chat RAG com PDFs locais")
st.write("Faça perguntas com base nos PDFs presentes na pasta `src/RAG FILES`.")


In [ ]:
def list_pdf_files() -> list[Path]:
    PDF_DIR.mkdir(parents=True, exist_ok=True)
    return sorted(PDF_DIR.glob("*.pdf"))


def build_documents(pdf_files: list[Path]) -> list[Document]:
    documents: list[Document] = []
    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        documents.extend(loader.load())
    return documents


def split_documents(documents: list[Document]) -> list[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
    )
    return text_splitter.split_documents(documents)


def docs_to_string(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


def build_pdf_signature(pdf_files: list[Path]) -> str:
    signature_base = "|".join(
        f"{pdf_file.name}:{pdf_file.stat().st_mtime_ns}:{pdf_file.stat().st_size}"
        for pdf_file in pdf_files
    )
    return hashlib.sha256(signature_base.encode("utf-8")).hexdigest()


In [ ]:
@st.cache_resource(show_spinner=False)
def load_llm() -> HuggingFacePipeline:
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)

    if torch.backends.mps.is_available():
        device = "mps"
        torch_dtype = torch.float16
    elif torch.cuda.is_available():
        device = "cuda"
        torch_dtype = torch.bfloat16
    else:
        device = "cpu"
        torch_dtype = torch.float32

    pipe = pipeline(
        "text-generation",
        model=LLM_MODEL_ID,
        tokenizer=tokenizer,
        torch_dtype=torch_dtype,
        device=device if device != "cuda" else None,
        device_map="auto" if device == "cuda" else None,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=True,
        return_full_text=False,
        repetition_penalty=1.1,
    )
    return HuggingFacePipeline(pipeline=pipe)


@st.cache_resource(show_spinner=False)
def load_embeddings() -> HuggingFaceEmbeddings:
    model_kwargs = {"device": "cpu"}
    encode_kwargs = {"normalize_embeddings": True}
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_ID,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs,
    )


In [ ]:
@st.cache_resource(show_spinner=False)
def build_vector_store(pdf_signature: str, pdf_paths: tuple[str, ...]) -> Chroma:
    del pdf_signature
    pdf_files = [Path(path) for path in pdf_paths]
    documents = build_documents(pdf_files)
    splits = split_documents(documents)
    embeddings = load_embeddings()

    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR),
    )

    existing_ids = vector_store.get().get("ids", [])
    if existing_ids:
        vector_store.delete(ids=existing_ids)

    ids = [f"chunk-{index}" for index in range(len(splits))]
    vector_store.add_documents(documents=splits, ids=ids)
    return vector_store


In [ ]:
def create_rag_chain(vector_store: Chroma, chat_history: StreamlitChatMessageHistory):
    llm = load_llm()

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Voce e um assistente que responde apenas com base no contexto recuperado dos PDFs. "
                "Responda sempre em portugues, de forma clara e objetiva. "
                "Se a resposta nao estiver no contexto, diga isso explicitamente.",
            ),
            MessagesPlaceholder(variable_name="history"),
            (
                "human",
                "Contexto:\n{context}\n\nPergunta: {question}",
            ),
        ]
    )

    def retrieve_context(payload: dict) -> str:
        question = payload["question"]
        retriever = vector_store.as_retriever(search_kwargs={"k": 4})
        docs = retriever.invoke(question)
        return docs_to_string(docs)

    chain = {
        "context": RunnableLambda(retrieve_context),
        "question": RunnableLambda(lambda payload: payload["question"]),
        "history": RunnableLambda(lambda payload: payload.get("history", [])),
    } | prompt | llm

    return RunnableWithMessageHistory(
        chain,
        lambda session_id: chat_history,
        input_messages_key="question",
        history_messages_key="history",
    )


In [ ]:
pdf_files = list_pdf_files()

with st.sidebar:
    st.subheader("Base RAG")
    st.write(f"Pasta monitorada: `{PDF_DIR.name}`")
    if pdf_files:
        st.write(f"PDFs encontrados: {len(pdf_files)}")
        for pdf_file in pdf_files:
            st.caption(pdf_file.name)
    else:
        st.warning("Nenhum PDF encontrado. Adicione arquivos em `src/RAG FILES`.")

    if st.button("Recarregar base vetorial"):
        build_vector_store.clear()
        st.cache_resource.clear()
        st.rerun()


if "chat_history" not in st.session_state:
    st.session_state.chat_history = StreamlitChatMessageHistory(key="rag_chat_messages")

if len(st.session_state.chat_history.messages) == 0:
    st.session_state.chat_history.add_ai_message(
        "Envie sua pergunta e eu responderei com base nos PDFs da pasta RAG FILES."
    )


if not pdf_files:
    for msg in st.session_state.chat_history.messages:
        st.chat_message(msg.type).write(msg.content)
    st.stop()


pdf_signature = build_pdf_signature(pdf_files)
pdf_paths = tuple(str(pdf_file) for pdf_file in pdf_files)

with st.spinner("Preparando a base RAG..."):
    st.session_state.vector_store = build_vector_store(pdf_signature, pdf_paths)
    conversational_rag_chain = create_rag_chain(
        st.session_state.vector_store,
        st.session_state.chat_history,
    )


for msg in st.session_state.chat_history.messages:
    st.chat_message(msg.type).write(msg.content)


if user_input := st.chat_input("Pergunte algo sobre os PDFs..."):
    st.chat_message("human").write(user_input)

    with st.spinner("Buscando resposta nos documentos..."):
        config = {"configurable": {"session_id": "rag_streamlit_session"}}
        response = conversational_rag_chain.invoke({"question": user_input}, config=config)
        clean_response = response.split("<|im_end|>")[0].strip()

    st.chat_message("ai").write(clean_response)
